# 02m — Hierarchical YOLOv8 Classification (Cascaded, multi×2) with Synthetic Data

**Project:** UREP 32-0210-250078 | Crack Classification

**Architecture:** 3 independent YOLOv8s-cls models in a cascade

## Hierarchy

```
                Input image
                     │
         Stage 1: crack vs no_crack
                     │
              ┌──────┴──────┐
            crack          no_crack
              │
      Stage 2: single vs multi
              │
         ┌────┴────┐
       single    multi
         │
  Stage 3: 4-way subtype
  {debonding, flexural, shear, others}
```

## Notes

* Same hyperparameters as `02e_training_yolo.ipynb` for all 3 stages.
* Same augmentations (rotation ±15°, fliplr, no flipud, no shear).
* **Experiment variant:** Stage 2 uses **multi×2** balancing rule: `target_single = target_multi = 2·N_multi`.
* Stages 1 and 3 use the standard ×6 cap rule on the smallest class.
* **Synthetic data:** 806 synthetic AutoCAD-generated crack images added to training
  (102 debonding/corrosion, 302 flexural, 402 shear). Val/test identical to baseline.
* Existing 70/15/15 split is reused — directly comparable to baseline and non-synthetic variant.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import numpy as np
from ultralytics import YOLO

import config
from src.dataset import prepare_synthetic_split
from src.evaluation import evaluate_predictions
from src.device import set_seed
from src.hierarchical import (
    STAGE1_CLASSES, STAGE2_CLASSES, STAGE3_CLASSES,
    build_yolo_stage_dirs, materialize_balanced_yolo_train,
    cascade_predict, cascaded_to_orig,
    hierarchical_pr_f1, per_stage_confusion_matrices, error_attribution,
    SUPPORTED_EXTENSIONS,
)

# Reproducibility
set_seed(config.RANDOM_SEED)

OUTPUT_DIR = os.path.join(config.OUTPUT_DIR, "yolo_hier_multix2_synthetic")
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)

# Materialize synthetic split
SPLIT = config.SPLIT_SYNTHETIC_DIR
prepare_synthetic_split()

# Separate hierarchical dataset folder (materialize modifies files in-place)
HIER_ROOT = os.path.join(config.DATA_DIR, "hierarchical_yolo_multix2_synthetic")

print(f"Model: YOLOv8s-cls (cascaded, multi×2, with Synthetic Data)")
print(f"Source split: {SPLIT}")
print(f"Hier root:    {HIER_ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")

## Build per-stage folder trees

Materializes 3 folder trees from the existing flat split via symlinks (or copies on Windows w/o developer mode). Re-runnable: skips files that already exist.

In [ ]:
stage_counts = build_yolo_stage_dirs(SPLIT, HIER_ROOT)

## Rebalance training folders

* **Stage 1:** ×6 cap rule on smallest of {crack, no_crack}
* **Stage 2:** `target_single = target_multi = 2·N_multi` (downsample single, oversample multi ×2)
* **Stage 3:** ×6 cap rule on smallest of {debonding, flexural, shear, others}

Val/test folders are left untouched.

In [ ]:
stage1_balance = materialize_balanced_yolo_train(
    os.path.join(HIER_ROOT, "stage1"), stage="stage1", rule="cap6",
)
stage2_balance = materialize_balanced_yolo_train(
    os.path.join(HIER_ROOT, "stage2"), stage="stage2", rule="multix3",
    multi_factor=2,
)
stage3_balance = materialize_balanced_yolo_train(
    os.path.join(HIER_ROOT, "stage3"), stage="stage3", rule="cap6",
)

## Stage 1 — Train crack / no_crack classifier

Same hyperparameters as `02e_training_yolo.ipynb`.

In [ ]:
model_s1 = YOLO(config.YOLO_MODEL)
results_s1 = model_s1.train(
    data=os.path.join(HIER_ROOT, "stage1"),
    epochs=config.YOLO_EPOCHS,
    imgsz=config.YOLO_IMG_SIZE,
    batch=16,
    patience=config.YOLO_PATIENCE,
    lr0=config.YOLO_LR0,
    lrf=config.YOLO_LRF,
    dropout=config.YOLO_DROPOUT,
    optimizer="AdamW",
    degrees=15.0, fliplr=0.5, flipud=0.0, shear=0.0,
    seed=config.RANDOM_SEED,
    project=OUTPUT_DIR, name="stage1", exist_ok=True, verbose=True,
)

## Stage 2 — Train single / multi classifier

In [ ]:
model_s2 = YOLO(config.YOLO_MODEL)
results_s2 = model_s2.train(
    data=os.path.join(HIER_ROOT, "stage2"),
    epochs=config.YOLO_EPOCHS,
    imgsz=config.YOLO_IMG_SIZE,
    batch=16,
    patience=config.YOLO_PATIENCE,
    lr0=config.YOLO_LR0,
    lrf=config.YOLO_LRF,
    dropout=config.YOLO_DROPOUT,
    optimizer="AdamW",
    degrees=15.0, fliplr=0.5, flipud=0.0, shear=0.0,
    seed=config.RANDOM_SEED,
    project=OUTPUT_DIR, name="stage2", exist_ok=True, verbose=True,
)

## Stage 3 — Train 4-way subtype classifier

Classes: `debonding`, `flexural`, `shear`, `others`.

In [ ]:
model_s3 = YOLO(config.YOLO_MODEL)
results_s3 = model_s3.train(
    data=os.path.join(HIER_ROOT, "stage3"),
    epochs=config.YOLO_EPOCHS,
    imgsz=config.YOLO_IMG_SIZE,
    batch=16,
    patience=config.YOLO_PATIENCE,
    lr0=config.YOLO_LR0,
    lrf=config.YOLO_LRF,
    dropout=config.YOLO_DROPOUT,
    optimizer="AdamW",
    degrees=15.0, fliplr=0.5, flipud=0.0, shear=0.0,
    seed=config.RANDOM_SEED,
    project=OUTPUT_DIR, name="stage3", exist_ok=True, verbose=True,
)

## Cascaded inference on the test set

We iterate the **original** flat test split (no synthetic images) so that ground truth
stays in the original 6-class space and the resulting confusion matrix is directly comparable
to the flat YOLO baseline and the non-synthetic hierarchical variant.

In [ ]:
best_s1 = os.path.join(OUTPUT_DIR, "stage1", "weights", "best.pt")
best_s2 = os.path.join(OUTPUT_DIR, "stage2", "weights", "best.pt")
best_s3 = os.path.join(OUTPUT_DIR, "stage3", "weights", "best.pt")

model_s1 = YOLO(best_s1)
model_s2 = YOLO(best_s2)
model_s3 = YOLO(best_s3)


def _names_to_prob_vec(result, ordered_classes):
    """Reorder a YOLO classification result's probability vector to match
    the canonical class order used by src.hierarchical."""
    probs = result[0].probs.data.cpu().numpy()
    name_to_idx = {n: i for i, n in result[0].names.items()}
    return np.array([probs[name_to_idx[c]] for c in ordered_classes], dtype=np.float64)


from src.hierarchical import ORIG_TO_HIER

# Test from ORIGINAL split (no synthetic) for fair comparison
test_dir = os.path.join(config.SPLIT_DIR, "test")
y_true_paths, y_pred_paths = [], []
y_true_flat, y_pred_flat = [], []

for orig_cls in config.CLASS_NAMES:
    cls_dir = os.path.join(test_dir, orig_cls)
    if not os.path.isdir(cls_dir):
        continue
    files = [f for f in os.listdir(cls_dir)
             if os.path.splitext(f)[1].lower() in SUPPORTED_EXTENSIONS
             and not f.startswith("aug_")]
    true_path = ORIG_TO_HIER[orig_cls]

    for fname in files:
        path = os.path.join(cls_dir, fname)

        # Always run all 3 models so per-stage CMs include the right populations.
        r1 = model_s1.predict(path, verbose=False)
        r2 = model_s2.predict(path, verbose=False)
        r3 = model_s3.predict(path, verbose=False)

        p1 = _names_to_prob_vec(r1, STAGE1_CLASSES)
        p2 = _names_to_prob_vec(r2, STAGE2_CLASSES)
        p3 = _names_to_prob_vec(r3, STAGE3_CLASSES)

        pred_path = cascade_predict(p1, p2, p3, t1=0.5, t2=0.5)

        y_true_paths.append(true_path)
        y_pred_paths.append(pred_path)
        y_true_flat.append(orig_cls)
        y_pred_flat.append(cascaded_to_orig(*pred_path))

    print(f"  {orig_cls}: {len(files)} done")

print(f"\nTotal test predictions: {len(y_true_flat)}")

## Flat 6-class evaluation (directly comparable to baseline YOLO)

In [ ]:
y_true_idx = np.array([config.CLASS_NAMES.index(c) for c in y_true_flat])
y_pred_idx = np.array([config.CLASS_NAMES.index(c) for c in y_pred_flat])

metrics = evaluate_predictions(
    y_true_idx, y_pred_idx,
    output_dir=OUTPUT_DIR, model_name="yolo_hier_multix2_synth",
)

## Per-stage confusion matrices, hierarchical metrics, error attribution

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns

stage_cms = per_stage_confusion_matrices(y_true_paths, y_pred_paths)
h_metrics = hierarchical_pr_f1(y_true_paths, y_pred_paths)
err_attr  = error_attribution(y_true_paths, y_pred_paths)

print("Hierarchical precision/recall/F1:")
for k, v in h_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nError attribution:")
print(f"  Total:        {err_attr['total']}")
print(f"  Correct:      {err_attr['correct']}")
print(f"  Stage1 errs:  {err_attr['errors_by_stage']['stage1']}")
print(f"  Stage2 errs:  {err_attr['errors_by_stage']['stage2']}")
print(f"  Stage3 errs:  {err_attr['errors_by_stage']['stage3']}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key in zip(axes, ["stage1", "stage2", "stage3"]):
    if key not in stage_cms:
        ax.set_visible(False); continue
    info = stage_cms[key]
    sns.heatmap(info["cm"], annot=True, fmt="d", cmap="Blues",
                xticklabels=info["classes"], yticklabels=info["classes"], ax=ax)
    ax.set_title(f"{key}  ({info['cm'].sum()} samples)")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "per_stage_confusion_matrices.png"),
            dpi=150, bbox_inches="tight")
plt.show()

with open(os.path.join(OUTPUT_DIR, "hierarchical_metrics.json"), "w") as f:
    json.dump({
        "hierarchical": h_metrics,
        "error_attribution": err_attr,
        "flat": {
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
            "f1_weighted": metrics["f1_weighted"],
        },
    }, f, indent=2)
print(f"\nSaved hierarchical metrics to {OUTPUT_DIR}/hierarchical_metrics.json")